# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/barak-hub/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [8]:
import pandas as pd
import numpy as np

features = [
    "impressions_prev30",
    "clicks_prev30",
    "avg_position_prev30",
    "query_token_count",
    "query_char_count"
]

print("Selected historical features:")
for feature in features:
    print("-", feature)

print("\nTotal features:", len(features))

Selected historical features:
- impressions_prev30
- clicks_prev30
- avg_position_prev30
- query_token_count
- query_char_count

Total features: 5


In [9]:
# Example historical Google Search data
example_impressions = 1000
example_clicks = 20

# Calculate historical CTR
historical_ctr = example_clicks / example_impressions

print("Historical CTR:", historical_ctr)
print("Historical CTR (%):", historical_ctr * 100)

# Add CTR to our selected feature list
features.append("ctr_prev30")

print("\nTotal selected features:", len(features))

Historical CTR: 0.02
Historical CTR (%): 2.0

Total selected features: 6


In [10]:
import os

print("Dataset loaded:", "df" in globals())
print("Hugging Face token configured:",
      "HF_TOKEN" in os.environ)

Dataset loaded: False
Hugging Face token configured: False


In [11]:
from google.colab import userdata

hf_token = userdata.get("HF_TOKEN")

print("Hugging Face token available:", bool(hf_token))

Hugging Face token available: True


In [12]:
%pip -q install huggingface_hub pyarrow

from huggingface_hub import hf_hub_download
import pandas as pd

file_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_query_90d.parquet",
    repo_type="dataset",
    token=hf_token
)

df = pd.read_parquet(file_path)

print("Dataset loaded successfully!")
print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Dataset shape:", df.shape)

fact_content_query_90d.parquet: reconstructing file:   0%|          |  0.00B / 60.7MB            

fact_content_query_90d.parquet: downloading bytes:           |  0.00B            

Dataset loaded successfully!
Rows: 2414248
Columns: 21
Dataset shape: (2414248, 21)


In [13]:

# Select records with sufficient historical impressions
candidates = df[df["impressions_prev30"] >= 30].copy()

# Calculate historical CTR
candidates["ctr_prev30"] = (
    candidates["clicks_prev30"] /
    candidates["impressions_prev30"]
)

# Build the feature matrix
feature_columns = [
    "impressions_prev30",
    "clicks_prev30",
    "avg_position_prev30",
    "ctr_prev30",
    "query_token_count",
    "query_char_count"
]

X = candidates[feature_columns].copy()

print("Eligible records:", len(candidates))
print("Feature matrix shape:", X.shape)
print("\nMissing values by feature:")
print(X.isna().sum())


Eligible records: 390353
Feature matrix shape: (390353, 6)

Missing values by feature:
impressions_prev30     0
clicks_prev30          0
avg_position_prev30    0
ctr_prev30             0
query_token_count      0
query_char_count       0
dtype: int64


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [15]:

feature_notes = pd.DataFrame({
    "Feature": feature_columns,
    "Missing values": X.isna().sum().values,
    "Available before prediction": [
        "Yes", "Yes", "Yes",
        "Yes", "Yes", "Yes"
    ]
})

print(feature_notes.to_string(index=False))


            Feature  Missing values Available before prediction
 impressions_prev30               0                         Yes
      clicks_prev30               0                         Yes
avg_position_prev30               0                         Yes
         ctr_prev30               0                         Yes
  query_token_count               0                         Yes
   query_char_count               0                         Yes


In [16]:

feature_meanings = {
    "impressions_prev30": "Search impressions in the previous 30 days",
    "clicks_prev30": "Search clicks in the previous 30 days",
    "avg_position_prev30": "Average search ranking in the previous 30 days",
    "ctr_prev30": "Historical clicks divided by historical impressions",
    "query_token_count": "Number of words or tokens in the search query",
    "query_char_count": "Number of characters in the search query"
}

for feature, meaning in feature_meanings.items():
    print(f"{feature}: {meaning}")


impressions_prev30: Search impressions in the previous 30 days
clicks_prev30: Search clicks in the previous 30 days
avg_position_prev30: Average search ranking in the previous 30 days
ctr_prev30: Historical clicks divided by historical impressions
query_token_count: Number of words or tokens in the search query
query_char_count: Number of characters in the search query


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [18]:

# Check for identifiers and raw text in model inputs
sensitive_columns = [
    "client_hash_id",
    "content_hash_id",
    "query_hash_id",
    "query",
    "client_name",
    "content_url"
]

found_sensitive = [
    col for col in X.columns
    if col in sensitive_columns
]

print("Sensitive columns in feature matrix:")
print(found_sensitive)

if not found_sensitive:
    print("PASS: No listed identifiers or raw text in X.")
else:
    print("WARNING: Review sensitive features.")


Sensitive columns in feature matrix:
[]
PASS: No listed identifiers or raw text in X.


In [19]:

# Features that could leak future information
future_features = [
    "impressions_last30",
    "clicks_last30",
    "avg_position_last30",
    "impressions_90d",
    "clicks_90d",
    "avg_position_90d"
]

# Check whether any are in our model inputs
leaked_features = [
    col for col in X.columns
    if col in future_features
]

print("Selected model features:")
print(X.columns.tolist())

print("\nPotentially leaking features found:")
print(leaked_features)

if len(leaked_features) == 0:
    print("\nPASS: No listed future or full-window features in X.")
else:
    print("\nWARNING: Review potential data leakage!")


Selected model features:
['impressions_prev30', 'clicks_prev30', 'avg_position_prev30', 'ctr_prev30', 'query_token_count', 'query_char_count']

Potentially leaking features found:
[]

PASS: No listed future or full-window features in X.


In [20]:

excluded_features = {
    "clicks_last30": "Future-period clicks",
    "impressions_last30": "Future-period impressions",
    "avg_position_last30": "Future-period search position",
    "clicks_90d": "Overlaps the prediction period",
    "impressions_90d": "Overlaps the prediction period",
    "avg_position_90d": "Overlaps the prediction period"
}

print("EXCLUDED FEATURES AND REASONS\n")

for feature, reason in excluded_features.items():
    print(f"{feature}: {reason}")

print("\nTotal excluded features:", len(excluded_features))


EXCLUDED FEATURES AND REASONS

clicks_last30: Future-period clicks
impressions_last30: Future-period impressions
avg_position_last30: Future-period search position
clicks_90d: Overlaps the prediction period
impressions_90d: Overlaps the prediction period
avg_position_90d: Overlaps the prediction period

Total excluded features: 6


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.